# Treinamento YOLO — resíduos plásticos
Execute uma célula por vez. Este notebook não contém pesos treinados ou métricas prontas.

Antes: extraia **dataset_plasticos.zip** de APS.zip e coloque-o em **Meu Drive/APS/**.
O APS.zip completo tem cerca de 14,3 GB e contém cópias dos mesmos dados; prefira o ZIP interno de aproximadamente 4,7 GB.
Selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**. A oferta de GPU depende do Colab.

O comando `files.upload()` mostrado na captura somente faz upload. Para este volume, usamos o Drive.

Fontes: [Colab e arquivos](https://colab.research.google.com/notebooks/io.ipynb), [YOLO treino](https://docs.ultralytics.com/modes/train/), [TACO](https://github.com/pedropro/TACO).


## 1. Instalar as dependências do treinamento e conferir a GPU

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'ultralytics==8.4.191', 'PyYAML==6.0.3'], check=True)
import torch
print('PyTorch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Selecione GPU no ambiente de execução e execute novamente. O notebook não inicia treino em CPU silenciosamente.')
print('GPU:', torch.cuda.get_device_name(0))


## 2. Conectar o Drive e informar o caminho do arquivo

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ZIP_NO_DRIVE = '/content/drive/MyDrive/APS/dataset_plasticos.zip' #@param {type:"string"}
RESULTADOS_NO_DRIVE = '/content/drive/MyDrive/APS/resultados_yolo' #@param {type:"string"}
fonte = Path(ZIP_NO_DRIVE)
if not fonte.is_file():
    raise FileNotFoundError(f'Arquivo não encontrado: {fonte}. Confira o nome e a pasta no Drive.')
resultados = Path(RESULTADOS_NO_DRIVE)
resultados.mkdir(parents=True, exist_ok=True)
print('ZIP:', fonte.name, '| bytes:', fonte.stat().st_size)


## 3. Obter o código e copiar o ZIP para o disco da sessão
As anotações COCO originais fornecidas estão no repositório. A extração usa apenas fotos; os labels são reconstruídos pelas regras documentadas em treinamento/README.md.

In [ ]:
import os, shutil
repo = Path('/content/aps-plasticos')
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Eduuh1227/Projeto-Detector-de-Pl-sticos-.git', str(repo)], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Código utilizado:', commit)
zip_local = Path('/content/dataset_plasticos.zip')
if not zip_local.exists():
    if shutil.disk_usage('/content').free < fonte.stat().st_size + 6 * 1024**3:
        raise OSError('Libere espaço no disco da sessão antes de copiar/extrair os dados.')
    shutil.copyfile(fonte, zip_local)
elif zip_local.stat().st_size != fonte.stat().st_size:
    raise ValueError('Já existe outro ZIP local. Inicie uma sessão limpa para não misturar datasets.')
print('ZIP disponível no disco local da sessão.')


## 4. Preparar uma cópia nova do dataset
A política inclui embalagens e espuma plástica/isopor, sem excluir classes raras. Caixas na borda são interseccionadas com a imagem. A divisão recebida é preservada. O ZIP original fica intacto.

In [ ]:
from preparar_dataset_taco import preparar
from datetime import datetime, timezone
pasta_dados = Path('/content/dados_treinamento_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S'))
yaml_dataset = preparar(zip_local, pasta_dados, repo / 'dataset/taco/annotations.json')
print('YAML:', yaml_dataset)


## 5. Validar e revisar as anotações
O validador examina arquivos, caixas e duplicatas exatas. Isso não substitui revisão visual. Negativos precisam de atenção: ausência de uma classe selecionada no COCO não comprova ausência visual de plástico. Verifique também cenas semelhantes entre partições. Categorias ambíguas não são resolvidas automaticamente.

In [ ]:
from modelo.dados import validar_dataset
import json
dados_validados = validar_dataset(yaml_dataset, ('train','val','test'))
print(json.dumps(dados_validados.contagens, indent=2))
politica = json.loads((pasta_dados / 'conversao_taco.json').read_text())
print('Categorias incluídas:')
for categoria in politica['categorias_originais_incluidas']:
    print(categoria['id'], categoria['nome'], categoria['caixas'])
print('Caixas ajustadas aos limites:', politica['caixas_interseccionadas_com_a_imagem'])


In [ ]:
import cv2, matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
fotos = sorted((pasta_dados / 'images/train').glob('*.jpg'))
for ax, foto in zip(axes.flat, fotos[:6]):
    imagem = cv2.imread(str(foto))
    if imagem is None:
        raise ValueError(f'Não foi possível abrir {foto.name}')
    h, w = imagem.shape[:2]
    label = pasta_dados / 'labels/train' / (foto.stem + '.txt')
    for linha in label.read_text().splitlines():
        _, xc, yc, bw, bh = map(float, linha.split())
        cv2.rectangle(imagem, (round((xc-bw/2)*w), round((yc-bh/2)*h)),
                      (round((xc+bw/2)*w), round((yc+bh/2)*h)), (0,255,0), 3)
    ax.imshow(cv2.cvtColor(imagem, cv2.COLOR_BGR2RGB))
    ax.set_title(foto.name)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Amostra técnica de revisão. Examine também negativos e outras imagens; seis exemplos não validam todo o dataset.')


## 6. Treinar
Após revisar os dados, marque a confirmação. Esta célula baixa explicitamente **yolo11n.pt** como ponto de partida, depois faz fine-tuning. As épocas, resolução e batch abaixo são parâmetros iniciais, sem promessa de desempenho. Os checkpoints e registros são salvos no Drive. Se faltar memória GPU, diminua batch e execute uma nova tentativa registrada.

In [ ]:
ANOTACOES_REVISADAS = False #@param {type:"boolean"}
EPOCHS = 50 #@param {type:"integer"}
IMGSZ = 640 #@param {type:"integer"}
BATCH = 8 #@param {type:"integer"}
if not ANOTACOES_REVISADAS:
    raise RuntimeError('Revise as categorias, caixas e negativos; depois marque ANOTACOES_REVISADAS para iniciar.')
from ultralytics import YOLO
base = YOLO('yolo11n.pt') # Download explícito dos pesos base oficiais.
from modelo.treinamento import treinar
melhor = treinar(repo / 'yolo11n.pt', dataset=yaml_dataset, epochs=EPOCHS,
                imgsz=IMGSZ, batch=BATCH, device='0', seed=42,
                pasta_execucoes=resultados / 'execucoes', pasta_modelos=resultados / 'modelos')
(resultados / 'ultimo_modelo.txt').write_text(str(melhor))
shutil.copyfile(pasta_dados / 'conversao_taco.json', resultados / ('conversao_' + melhor.stem + '.json'))
print('Checkpoint treinado:', melhor)


## 7. Avaliar em val durante o desenvolvimento

In [ ]:
from modelo.avaliacao import avaliar
metricas_val = avaliar(melhor, dataset=yaml_dataset, split='val', device='0',
                      imgsz=IMGSZ, batch=BATCH, pasta_metricas=resultados / 'metricas')
print('Métricas reais da validação:', metricas_val)
print(metricas_val.read_text())


## 8. Teste final independente
Execute quando as decisões do experimento estiverem fixadas. Não escolha parâmetros pelos resultados de test. Nenhuma métrica é inventada se a avaliação falhar.

In [ ]:
EXECUTAR_TESTE_FINAL = False #@param {type:"boolean"}
if EXECUTAR_TESTE_FINAL:
    metricas_test = avaliar(melhor, dataset=yaml_dataset, split='test', device='0',
                           imgsz=IMGSZ, batch=BATCH, pasta_metricas=resultados / 'metricas')
    print(metricas_test.read_text())
else:
    print('Avaliação final adiada. Use val enquanto estiver ajustando o experimento.')


## 9. Baixar o modelo
O arquivo abaixo só existe após treinamento bem-sucedido. Ele também permanece no Drive. No site Vercel, os pesos precisam ser incluídos em uma nova implantação; enviar uma fotografia ao site não instala nem treina o modelo.

In [ ]:
from google.colab import files
if not Path(melhor).is_file():
    raise FileNotFoundError('O treinamento ainda não gerou o checkpoint.')
files.download(str(melhor))


## Ao encerrar
Confira no Drive os pesos, treinamento.json, conversao_*.json e os JSONs de métricas efetivamente calculadas. O disco /content é temporário. Se uma sessão for interrompida, os checkpoints já gravados no Drive ficam disponíveis; retomar um treino exige restaurar também o mesmo dataset e seus caminhos.

Este notebook foi preparado para o material fornecido e verificado quanto à sintaxe. **Não foi executado um treinamento em GPU nesta entrega.** A qualidade científica ainda depende da revisão dos dados e da avaliação real.
